# Bird vs Drone — Aerial Object Classification & Detection

---

### What this project is about

Birds and drones look remarkably similar at altitude. Both are small, both move fast, and both photograph poorly against bright sky. That similarity is the whole challenge — and it turns out to be genuinely useful to solve. A drone near an airport runway is a safety incident. A bird near a wind turbine is an ecological one. Automated aerial classification can support real-time alerts for both.

This notebook builds that classifier from scratch, then improves it using pretrained models, then adds bounding box detection on top. By the end we have a working system ready to plug into a Streamlit web app.

### Notebook structure

| Section | What happens |
|---------|-------------|
| 1 | Setup — packages, Drive mount, paths, imports |
| 2 | EDA — understand the data before touching any model |
| 3 | Custom CNN — a convolutional network built from scratch |
| 4 | Transfer Learning — ResNet50, MobileNetV2, EfficientNetB0 |
| 5 | Model Comparison — pick the best one for deployment |
| 6 | YOLOv8 — add bounding box detection (optional) |

### Dataset

**Classification dataset** — 3,319 RGB `.jpg` images split across train / valid / test:
- train: 1,414 bird + 1,248 drone
- valid: 217 bird + 225 drone  
- test: 121 bird + 94 drone

**Detection dataset** — 3,319 images with YOLOv8 bounding box annotations (`.txt` files). Each line: `class_id cx cy width height` — all coordinates normalised to [0, 1].

> **Before running:** Go to `Runtime → Change runtime type → T4 GPU → Save`

---
## Section 1 — Setup

Three things to sort before any real work: get the extra packages Colab doesn't include by default, connect to Google Drive so we can read the dataset and write model files that survive a session reset, and define all paths in one place so there's only one cell to update if anything moves.

The packages we need on top of what Colab already has:
- `ultralytics` — the YOLOv8 library
- `seaborn` — cleaner statistical plots
- `scikit-learn` — confusion matrices and classification reports

In [ ]:
!pip install -q ultralytics seaborn scikit-learn
print("Done.")

### 1.1 Mount Google Drive

Colab sessions reset every few hours. Anything saved only to `/content/` disappears. Mounting Drive means trained model files persist — which matters a lot when a training run takes 30 minutes.

A popup will appear asking for permission. Allow it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

### 1.2 Paths & Constants

Your Google Drive structure looks like this (confirmed from your Drive):

```
MyDrive/
  aerial_project/
    classification_dataset/
      train/
        bird/
        drone/
      valid/
        bird/
        drone/
      test/
        bird/
        drone/
    object_detection_Dataset/
      train/
        images/
        labels/
      valid/
        images/
        labels/
      test/
        images/
        labels/
```

**Note:** Your splits are lowercase (`train`, `valid`, `test`) — not uppercase. This is already handled in the code below. If your folder names differ, update `CLASSIF_DIR` or `DETECT_DIR` here.

In [ ]:
import os

# ── Your Drive folder — change this only if your folder has a different name ──
BASE_DRIVE  = '/content/drive/MyDrive/aerial_project'

CLASSIF_DIR = os.path.join(BASE_DRIVE, 'classification_dataset')
DETECT_DIR  = os.path.join(BASE_DRIVE, 'object_detection_Dataset')

# Colab-local folders for outputs and saved models
# (copied to Drive at the end of Section 5 and Section 6)
OUTPUT_DIR  = '/content/outputs'
MODEL_DIR   = '/content/models'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR,  exist_ok=True)

# ── Dataset structure (confirmed from your Drive) ──
# Splits are lowercase: train / valid / test
# Class folders inside each split: bird / drone
SPLITS      = ['train', 'valid', 'test']   # lowercase — matches your Drive
CLASS_NAMES = ['bird', 'drone']
IMG_SIZE    = (224, 224)   # standard for ImageNet-pretrained models
BATCH_SIZE  = 32

# Quick check — both dataset folders should print Found
print("Path check:")
for path, label in [(CLASSIF_DIR, 'Classification dataset'),
                    (DETECT_DIR,  'Detection dataset')]:
    status = '✅  Found' if os.path.exists(path) else '❌  NOT FOUND'
    print(f"  {label}: {status}")
    print(f"     {path}")

### 1.3 Imports

All imports in one cell — no hunting through the notebook for where something was defined. We set random seeds for NumPy and TensorFlow so results are reproducible if you re-run a cell.

In [ ]:
import random, json, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, BatchNormalization,
    Dropout, Dense, GlobalAveragePooling2D
)
from tensorflow.keras.applications import ResNet50, MobileNetV2, EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {len(tf.config.list_physical_devices('GPU')) > 0}")

---
## Section 2 — Exploratory Data Analysis

There's a temptation to skip EDA and go straight to model building. That's usually a mistake. A few minutes of data inspection routinely surfaces problems — wrong folder structure, mislabelled images, extreme class imbalance, resolution mismatches — that would otherwise cause mysterious training failures.

We want to know:
- How many images per class, per split? Is there a meaningful imbalance?
- What do the images actually look like? Any obvious data quality issues?
- What sizes are the raw images? How much are we distorting them by resizing to 224×224?
- Are the pixel distributions between classes different enough to give the model a signal?
- Does the augmentation look reasonable, or is it destroying useful features?

### 2.1 Image Counts

We walk the folder tree and count `.jpg/.jpeg/.png` files per class per split. The imbalance ratio between bird and drone matters — anything under 1.5x is mild enough that augmentation handles it. Higher than that and we'd need to pass `class_weight` to `model.fit()`.

In [ ]:
counts = {}
print("Image counts per split:\n")

for split in SPLITS:
    counts[split] = {}
    for cls in CLASS_NAMES:
        path = os.path.join(CLASSIF_DIR, split, cls)
        n = len([f for f in os.listdir(path)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]) if os.path.exists(path) else 0
        counts[split][cls] = n
        print(f"  {split:5s} / {cls:5s} : {n} images")
    total = sum(counts[split].values())
    print(f"  {split:5s} total   : {total}\n")

b = counts['train']['bird']
d = counts['train']['drone']
ratio = max(b, d) / (min(b, d) + 1e-8)
print(f"Train imbalance → bird : drone = {b} : {d}  ({ratio:.2f}x)")
print("Mild imbalance." if ratio < 1.5 else "Notable imbalance — consider class_weight.")

### 2.2 Class Distribution Bar Chart

The same numbers visualised. The key thing to check here is whether valid and test have a similar ratio to train — if test is much heavier on one class, the accuracy number at evaluation will be misleading.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
colors = ['#4C72B0', '#DD8452']

for i, split in enumerate(SPLITS):
    vals = [counts[split].get(cls, 0) for cls in CLASS_NAMES]
    bars = axes[i].bar(CLASS_NAMES, vals, color=colors, width=0.5, edgecolor='white')
    axes[i].set_title(f'{split.capitalize()} Set', fontsize=13, fontweight='bold')
    axes[i].set_ylabel('Image Count')
    axes[i].set_ylim(0, max(vals) * 1.3)
    axes[i].grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 12, str(val),
                     ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Class Distribution — Train / Valid / Test', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

### 2.3 Sample Images

Five random images from each class. This is worth taking a moment to actually look at — not just confirming the cell ran. Things to notice: how much sky vs background is in each image, how large the object appears relative to the frame, whether some images are clearly harder than others.

In [ ]:
def get_sample_paths(base_dir, split, cls, n=5):
    folder = os.path.join(base_dir, split, cls)
    files  = [f for f in os.listdir(folder)
               if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    return [os.path.join(folder, f) for f in random.sample(files, min(n, len(files)))]

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for row, cls in enumerate(CLASS_NAMES):
    for col, path in enumerate(get_sample_paths(CLASSIF_DIR, 'train', cls, 5)):
        img = Image.open(path).convert('RGB')
        axes[row][col].imshow(img)
        axes[row][col].axis('off')
        axes[row][col].set_title(f'{cls}  {img.size[0]}x{img.size[1]}', fontsize=9)

plt.suptitle('Sample Training Images — Bird (top) vs Drone (bottom)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

### 2.4 Image Dimensions

All our models expect 224x224 input. The scatter plot shows the raw resolution of a random sample from each class. If most images are already close to 224x224, resizing introduces minimal distortion. If images are mostly 640x640 or 1280x720, we're squashing them significantly — in that case padding + cropping would be better than stretching, but for this dataset resizing is fine.

In [ ]:
widths, heights, cls_labels = [], [], []

for cls in CLASS_NAMES:
    folder = os.path.join(CLASSIF_DIR, 'train', cls)
    files  = [f for f in os.listdir(folder)
               if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    for f in random.sample(files, min(80, len(files))):
        try:
            img = Image.open(os.path.join(folder, f))
            widths.append(img.size[0])
            heights.append(img.size[1])
            cls_labels.append(cls)
        except Exception:
            pass

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
clrs = ['#4C72B0', '#DD8452']

for cls, color in zip(CLASS_NAMES, clrs):
    idx = [i for i, c in enumerate(cls_labels) if c == cls]
    axes[0].scatter([widths[i] for i in idx], [heights[i] for i in idx],
                    alpha=0.4, label=cls, color=color, s=20)
axes[0].axvline(224, color='red', linestyle='--', alpha=0.6, label='224 target')
axes[0].axhline(224, color='red', linestyle='--', alpha=0.6)
axes[0].set_xlabel('Width (px)'); axes[0].set_ylabel('Height (px)')
axes[0].set_title('Raw Image Dimensions', fontweight='bold')
axes[0].legend()

axes[1].hist(widths,  bins=30, color='#4C72B0', alpha=0.6, label='Width')
axes[1].hist(heights, bins=30, color='#DD8452', alpha=0.6, label='Height')
axes[1].axvline(224, color='red', linestyle='--', label='224px target')
axes[1].set_title('Width & Height Distribution', fontweight='bold')
axes[1].set_xlabel('Pixels'); axes[1].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/image_dimensions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Width  → mean {np.mean(widths):.0f}px | min {min(widths)} | max {max(widths)}")
print(f"Height → mean {np.mean(heights):.0f}px | min {min(heights)} | max {max(heights)}")

### 2.5 Pixel Intensity Distributions

This shows how red, green, and blue channel values are distributed for each class. If the histograms look very similar between bird and drone, the model has to rely on shape and texture rather than colour — which is harder. If there's a visible difference (drones tend to be photographed against clear sky, birds often against foliage), colour becomes a useful signal.

We downsample to 64x64 before reading pixel values — just for speed. We only care about the distribution shape here, not exact pixel counts.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ch_names  = ['Red', 'Green', 'Blue']
ch_colors = ['red', 'green', 'blue']

for ax, cls in zip(axes, CLASS_NAMES):
    folder  = os.path.join(CLASSIF_DIR, 'train', cls)
    files   = [f for f in os.listdir(folder)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    ch_data = {0: [], 1: [], 2: []}
    for f in random.sample(files, min(30, len(files))):
        arr = np.array(
            Image.open(os.path.join(folder, f)).convert('RGB').resize((64, 64)))
        for c in range(3):
            ch_data[c].extend(arr[:, :, c].flatten().tolist())
    for c, (name, col) in enumerate(zip(ch_names, ch_colors)):
        ax.hist(ch_data[c], bins=50, alpha=0.5, color=col, label=name, density=True)
    ax.set_title(f'Pixel Distribution — {cls.capitalize()}', fontweight='bold')
    ax.set_xlabel('Pixel Value (0-255)'); ax.set_ylabel('Density'); ax.legend()

plt.suptitle('RGB Channel Distributions by Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/pixel_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

### 2.6 Augmentation Preview

Augmentation is how we get more training signal out of a small dataset. Rather than the model seeing the same 2,600 images every epoch, it sees randomly transformed versions — rotated, flipped, slightly zoomed, brighter or darker. The model has to learn features that survive those transformations, which forces it to generalise rather than memorise.

What we apply:
- **Rotation ±20°** — birds and drones appear at any angle
- **Width/height shift ±10%** — slight position changes
- **Zoom ±15%** — different altitudes, different apparent sizes
- **Horizontal flip** — direction of flight doesn't help classify
- **Brightness ±20%** — different times of day, different cloud cover

The preview shows 9 augmented versions of one image. The augmentation should look noticeable but not extreme — if the object becomes unrecognisable in most versions, it's too aggressive.

In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array

aug_gen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

sample_path = get_sample_paths(CLASSIF_DIR, 'train', 'drone', 1)[0]
orig = load_img(sample_path, target_size=IMG_SIZE)
arr  = img_to_array(orig).reshape((1,) + img_to_array(orig).shape)
aug_it = aug_gen.flow(arr, batch_size=1)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes[0][0].imshow(orig)
axes[0][0].set_title('Original', fontweight='bold', color='green')
axes[0][0].axis('off')

for r, c in [(0,1),(0,2),(0,3),(0,4),(1,0),(1,1),(1,2),(1,3),(1,4)]:
    axes[r][c].imshow(next(aug_it)[0].astype(np.uint8))
    axes[r][c].set_title(f'Aug #{r*5+c}', fontsize=9)
    axes[r][c].axis('off')

plt.suptitle('Augmentation Previews — Original vs 9 Random Transforms (Drone sample)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/augmentation_preview.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

### 2.7 Data Generators

Keras `ImageDataGenerator` reads images from disk in batches, applies augmentation on the fly, and normalises pixel values from [0, 255] to [0, 1]. It figures out class labels automatically from the subfolder names (`bird` → 0, `drone` → 1 alphabetically).

We wrap generator creation in a `make_generators()` function rather than building them once — each model section calls it fresh so generators start from the beginning of the dataset rather than wherever a previous training run left off.

Validation and test generators get **no augmentation** — only rescaling. Augmenting evaluation data would change the distribution we're measuring against, which makes the metrics meaningless.

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

def make_generators():
    """Fresh generators — call before each model to reset state."""
    train_gen = train_datagen.flow_from_directory(
        os.path.join(CLASSIF_DIR, 'train'),
        target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='binary', shuffle=True, seed=42
    )
    val_gen = val_test_datagen.flow_from_directory(
        os.path.join(CLASSIF_DIR, 'valid'),
        target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='binary', shuffle=False
    )
    test_gen = val_test_datagen.flow_from_directory(
        os.path.join(CLASSIF_DIR, 'test'),
        target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='binary', shuffle=False
    )
    return train_gen, val_gen, test_gen

train_gen, val_gen, test_gen = make_generators()
print(f"Class indices : {train_gen.class_indices}")
print(f"Train batches : {len(train_gen)}")
print(f"Val batches   : {len(val_gen)}")
print(f"Test batches  : {len(test_gen)}")

---
## Section 3 — Custom CNN

We start with a network built entirely from scratch — no pretrained weights, no borrowed features. This is the baseline. It tells us what's achievable from our ~2,600 training images alone, and gives us a reference point for judging how much the pretrained models actually help.

### Architecture

Three convolutional blocks with increasing filter depth (32 → 64 → 128). Each block follows the same pattern:
- Two `Conv2D` layers with ReLU activation
- `BatchNormalization` after each conv — keeps activations in a reasonable range, speeds up training
- `MaxPooling2D` at the end — halves spatial dimensions, forces the network to find more compact representations
- `Dropout` — randomly zeroes out neurons during training, one of the more reliable ways to reduce overfitting on small datasets

The classifier head uses `GlobalAveragePooling2D` rather than `Flatten`. Flattening a 28×28×128 feature map gives 100,352 values feeding into a dense layer — that's a lot of parameters and a strong overfitting risk. GAP averages each feature map to a single number, giving 128 values instead. Much leaner, still effective.

Single sigmoid output neuron: values above 0.5 → drone, below 0.5 → bird.

### 3.1 Shared Helper Functions

Three functions used by all model sections:

**`plot_history()`** — plots accuracy and loss over epochs for both train and validation. The gap between the two curves is the clearest signal of overfitting.

**`evaluate_model()`** — runs the trained model on the held-out test set, prints accuracy/precision/recall/F1, draws a confusion matrix, and prints the full per-class classification report. The confusion matrix shows *which* errors the model makes, not just how many.

**`make_callbacks()`** — three training callbacks:
- `EarlyStopping` — stops training when val_loss stops improving (patience=8 epochs), then restores the best weights seen during the run
- `ModelCheckpoint` — saves weights to disk whenever val_accuracy improves, so we have the best checkpoint even if training continues past the peak
- `ReduceLROnPlateau` — halves the learning rate when val_loss stalls for 4 epochs; often unlocks a few more accuracy points late in training

In [ ]:
def plot_history(history, model_name, save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history.history['accuracy'],     label='Train',      color='#2196F3', lw=2)
    axes[0].plot(history.history['val_accuracy'], label='Validation', color='#FF5722', lw=2, ls='--')
    axes[0].set_title(f'{model_name} — Accuracy', fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(history.history['loss'],     label='Train',      color='#2196F3', lw=2)
    axes[1].plot(history.history['val_loss'], label='Validation', color='#FF5722', lw=2, ls='--')
    axes[1].set_title(f'{model_name} — Loss', fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def evaluate_model(model, test_gen, model_name, save_prefix):
    results = model.evaluate(test_gen, verbose=0)
    test_gen.reset()
    y_pred  = (model.predict(test_gen, verbose=0) > 0.5).astype(int).flatten()
    y_true  = test_gen.classes

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f'{model_name} — Confusion Matrix', fontweight='bold')
    ax.set_ylabel('True Label'); ax.set_xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/{save_prefix}_cm.png', dpi=150, bbox_inches='tight')
    plt.show()

    prec = float(results[2])
    rec  = float(results[3])
    f1   = 2 * prec * rec / (prec + rec + 1e-8)

    print(f"\n{'='*50}")
    print(f"  {model_name} — Test Results")
    print(f"{'='*50}")
    print(f"  Accuracy  : {results[1]:.4f}  ({results[1]*100:.1f}%)")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print()
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
    return {'model': model_name,
            'test_accuracy': float(results[1]),
            'test_precision': prec,
            'test_recall': rec,
            'test_f1': f1}


def make_callbacks(save_prefix):
    return [
        EarlyStopping(monitor='val_loss', patience=8,
                      restore_best_weights=True, verbose=1),
        ModelCheckpoint(f'{MODEL_DIR}/{save_prefix}_best.keras',
                        monitor='val_accuracy', save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=4, min_lr=1e-7, verbose=1)
    ]

print("Helper functions ready.")

### 3.2 Build the CNN

In [ ]:
def build_custom_cnn(input_shape=(224, 224, 3)):
    model = Sequential([
        # Block 1 — picks up edges and basic colour gradients
        Conv2D(32, (3,3), activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        Conv2D(32, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2,2)),   # 224x224 → 112x112
        Dropout(0.25),

        # Block 2 — picks up textures and simple shapes
        Conv2D(64, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(64, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2,2)),   # 112x112 → 56x56
        Dropout(0.25),

        # Block 3 — higher-level object parts
        Conv2D(128, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(128, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2,2)),   # 56x56 → 28x28
        Dropout(0.3),

        # Classifier head
        GlobalAveragePooling2D(),  # 28x28x128 → 128
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.4),
        Dense(1, activation='sigmoid')  # probability of drone
    ], name='Custom_CNN')
    return model

cnn_model = build_custom_cnn()
cnn_model.summary()

### 3.3 Train

Adam optimiser with learning rate 1e-3. Binary crossentropy loss — standard for single sigmoid output. We track precision and recall directly during training so the F1 components are available without a separate evaluation pass.

Max epochs is 50 but EarlyStopping will usually cut this short.

In [ ]:
cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

train_gen, val_gen, test_gen = make_generators()

print("Training Custom CNN...")
t0 = time.time()
history_cnn = cnn_model.fit(
    train_gen, epochs=50, validation_data=val_gen,
    callbacks=make_callbacks('cnn'), verbose=1
)
print(f"Done in {(time.time()-t0)/60:.1f} min")

### 3.4 Evaluate & Save

Training curves show whether the model overfit (big train/val gap) or was still improving when EarlyStopping fired (both curves still going up). The confusion matrix shows whether the model's mistakes are symmetric or skewed toward one class.

In [ ]:
plot_history(history_cnn, 'Custom CNN',
             save_path=f'{OUTPUT_DIR}/cnn_training_curves.png')

cnn_metrics = evaluate_model(cnn_model, test_gen, 'Custom CNN', 'cnn')

cnn_model.save(f'{MODEL_DIR}/custom_cnn_final.keras')
with open(f'{OUTPUT_DIR}/cnn_metrics.json', 'w') as f:
    json.dump(cnn_metrics, f, indent=2)
print("Custom CNN saved.")

---
## Section 4 — Transfer Learning

Transfer learning reuses a model trained on ImageNet (1.2 million images, 1000 classes) and adapts it to our task. The reasoning is that features learned on that scale — edges, textures, shapes, object parts — are general enough to be useful for aerial classification, even though birds and drones were never in ImageNet.

### Two-phase training

We use the same approach for all three models:

**Phase 1** — Freeze the entire pretrained base. Only the new classification head trains. Learning rate is 1e-3 because the head starts from random weights and needs to move quickly. This phase establishes roughly correct features without destroying the pretrained weights.

**Phase 2** — Unfreeze the top N layers of the base. Train with learning rate 1e-5 — much lower, because these weights are already good and we only want small adjustments to adapt them to our specific images. Using a high LR here would overwrite the ImageNet training and hurt performance.

### Why these three models?

| Model | Parameters | Reason included |
|-------|-----------|------------------|
| ResNet50 | 25.6M | Industry benchmark for transfer learning |
| MobileNetV2 | 3.4M | Fast inference — practical for Streamlit deployment |
| EfficientNetB0 | 5.3M | Compound scaling, usually best accuracy per parameter |

In [ ]:
def build_tl_model(base_fn, name, input_shape=(224, 224, 3)):
    """
    Wraps a pretrained base with a binary classification head:
    GlobalAveragePooling → Dense(256) → BatchNorm → Dropout(0.4) → Dense(1, sigmoid)
    """
    base = base_fn(weights='imagenet', include_top=False, input_shape=input_shape)
    base.trainable = False  # frozen for Phase 1
    x   = GlobalAveragePooling2D()(base.output)
    x   = Dense(256, activation='relu')(x)
    x   = BatchNormalization()(x)
    x   = Dropout(0.4)(x)
    out = Dense(1, activation='sigmoid')(x)
    return Model(inputs=base.input, outputs=out, name=name), base


def compile_model(model, lr):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall')]
    )


def train_two_phase(model, base, train_gen, val_gen,
                    unfreeze_layers, save_prefix, lr1=1e-3, lr2=1e-5):
    # Phase 1 — head only
    compile_model(model, lr1)
    print(f"Phase 1 — head only (base frozen), lr={lr1}")
    h1 = model.fit(train_gen, epochs=20, validation_data=val_gen,
                   callbacks=make_callbacks(f'{save_prefix}_p1'), verbose=1)

    # Phase 2 — fine-tune top layers
    base.trainable = True
    for layer in base.layers[:-unfreeze_layers]:
        layer.trainable = False
    compile_model(model, lr2)
    print(f"\nPhase 2 — top {unfreeze_layers} layers unfrozen, lr={lr2}")
    h2 = model.fit(train_gen, epochs=20, validation_data=val_gen,
                   callbacks=make_callbacks(f'{save_prefix}_p2'), verbose=1)
    return h1, h2

print("Transfer learning helpers ready.")

### 4.1 ResNet50

ResNet50 introduced residual connections — shortcut paths that let gradients flow directly through the network without passing through every layer. Before this, training networks deeper than ~20 layers reliably caused gradients to vanish. ResNet made 50, 101, and even 152-layer networks trainable. It's still the standard benchmark for transfer learning comparisons.

We unfreeze the last 30 layers for fine-tuning — these are the highest-level feature detectors that are most specific to the original training data and most likely to benefit from adaptation.

In [ ]:
train_gen, val_gen, test_gen = make_generators()
resnet_model, resnet_base = build_tl_model(ResNet50, 'ResNet50_BirdDrone')
print(f"ResNet50 — {resnet_model.count_params():,} total params")

t0 = time.time()
h_rn_p1, h_rn_p2 = train_two_phase(
    resnet_model, resnet_base, train_gen, val_gen,
    unfreeze_layers=30, save_prefix='resnet50'
)
print(f"Done in {(time.time()-t0)/60:.1f} min")

plot_history(h_rn_p1, 'ResNet50 Phase 1', f'{OUTPUT_DIR}/resnet50_p1_curves.png')
plot_history(h_rn_p2, 'ResNet50 Phase 2', f'{OUTPUT_DIR}/resnet50_p2_curves.png')
resnet_metrics = evaluate_model(resnet_model, test_gen, 'ResNet50', 'resnet50')
resnet_model.save(f'{MODEL_DIR}/resnet50_final.keras')
print("ResNet50 saved.")

### 4.2 MobileNetV2

MobileNetV2 uses depthwise separable convolutions — it splits the standard convolution operation into two cheaper steps: one that filters each channel independently, one that mixes channels. The result is about 8-9x fewer multiply-add operations than a standard convolution for the same output size. At 3.4M parameters it's roughly 7x smaller than ResNet50.

For deployment that matters. The Streamlit app needs to load the model on startup and run predictions per upload — MobileNetV2 does both faster than the other two options.

In [ ]:
train_gen, val_gen, test_gen = make_generators()
mobile_model, mobile_base = build_tl_model(MobileNetV2, 'MobileNetV2_BirdDrone')
print(f"MobileNetV2 — {mobile_model.count_params():,} total params")

t0 = time.time()
h_mv_p1, h_mv_p2 = train_two_phase(
    mobile_model, mobile_base, train_gen, val_gen,
    unfreeze_layers=20, save_prefix='mobilenetv2'
)
print(f"Done in {(time.time()-t0)/60:.1f} min")

plot_history(h_mv_p1, 'MobileNetV2 Phase 1', f'{OUTPUT_DIR}/mobilenetv2_p1_curves.png')
plot_history(h_mv_p2, 'MobileNetV2 Phase 2', f'{OUTPUT_DIR}/mobilenetv2_p2_curves.png')
mobile_metrics = evaluate_model(mobile_model, test_gen, 'MobileNetV2', 'mobilenetv2')
mobile_model.save(f'{MODEL_DIR}/mobilenetv2_final.keras')
print("MobileNetV2 saved.")

### 4.3 EfficientNetB0

EfficientNet (2019) noticed that most architecture improvements scale only one dimension — depth, or width, or resolution — independently. Compound scaling scales all three together using a fixed ratio found by neural architecture search. EfficientNetB0 is the smallest variant and is frequently the most accurate per parameter count in comparisons.

It's a bit sensitive to learning rate. We use 5e-4 in Phase 1 rather than 1e-3 — slightly slower start, but more stable.

In [ ]:
train_gen, val_gen, test_gen = make_generators()
effnet_model, effnet_base = build_tl_model(EfficientNetB0, 'EfficientNetB0_BirdDrone')
print(f"EfficientNetB0 — {effnet_model.count_params():,} total params")

t0 = time.time()
h_en_p1, h_en_p2 = train_two_phase(
    effnet_model, effnet_base, train_gen, val_gen,
    unfreeze_layers=20, save_prefix='efficientnetb0',
    lr1=5e-4, lr2=1e-5
)
print(f"Done in {(time.time()-t0)/60:.1f} min")

plot_history(h_en_p1, 'EfficientNetB0 Phase 1', f'{OUTPUT_DIR}/efficientnetb0_p1_curves.png')
plot_history(h_en_p2, 'EfficientNetB0 Phase 2', f'{OUTPUT_DIR}/efficientnetb0_p2_curves.png')
effnet_metrics = evaluate_model(effnet_model, test_gen, 'EfficientNetB0', 'efficientnetb0')
effnet_model.save(f'{MODEL_DIR}/efficientnetb0_final.keras')
print("EfficientNetB0 saved.")

---
## Section 5 — Model Comparison

Four models trained. Now we figure out which one to ship.

We rank by **F1 score** rather than raw accuracy. Accuracy is fine when classes are balanced, but it can be gamed by a model that leans heavily toward the majority class. F1 is the harmonic mean of precision and recall — it punishes models that do well on one metric at the expense of the other. For a security application like drone detection, both matter: you don't want false alarms (low precision) and you really don't want missed drones (low recall).

The radar chart shows all four metrics simultaneously for each model — a balanced, high-scoring model fills out more of the chart.

In [ ]:
import pandas as pd

all_metrics = [cnn_metrics, resnet_metrics, mobile_metrics, effnet_metrics]
df = pd.DataFrame(all_metrics).set_index('model')

print("\n" + "="*60)
print(" MODEL COMPARISON")
print("="*60)
print(df.round(4).to_string())
print("="*60)

with open(f'{OUTPUT_DIR}/tl_metrics.json', 'w') as f:
    json.dump([resnet_metrics, mobile_metrics, effnet_metrics], f, indent=2)

In [ ]:
# Bar charts — Accuracy and F1 side by side
model_names = list(df.index)
colors = ['#607D8B', '#2196F3', '#4CAF50', '#FF9800']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric, title in zip(axes,
                              ['test_accuracy', 'test_f1'],
                              ['Test Accuracy', 'Test F1 Score']):
    vals = df[metric].values
    bars = ax.bar(model_names, vals, color=colors, edgecolor='white', width=0.55)
    ax.set_ylim(0, 1.12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel('Score')
    ax.tick_params(axis='x', rotation=12)
    ax.grid(axis='y', alpha=0.3)
    ax.axhline(0.9, color='red', linestyle='--', alpha=0.4, label='90%')
    ax.legend(fontsize=8)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('All Models — Test Set Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/model_comparison_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

In [ ]:
# Radar chart — all four metrics at once
metrics_cols  = ['test_accuracy', 'test_precision', 'test_recall', 'test_f1']
metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1']
N      = len(metrics_cols)
angles = [n / float(N) * 2 * np.pi for n in range(N)] + [0]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for (mname, row), color in zip(df.iterrows(), colors):
    vals = [row[c] for c in metrics_cols] + [row[metrics_cols[0]]]
    ax.plot(angles, vals, 'o-', linewidth=2, label=mname, color=color)
    ax.fill(angles, vals, alpha=0.07, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.6, 0.7, 0.8, 0.9, 1.0])
ax.set_yticklabels(['60%','70%','80%','90%','100%'], fontsize=8)
ax.set_title('Performance Radar — All Models', fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.4, 1.1))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/model_radar.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

### 5.1 Select Best Model & Copy to Drive

The best model by F1 gets saved to Drive as `best_model.keras` — a consistent filename so `app.py` always knows what to load regardless of which model actually won. All output plots get copied to Drive too so they persist after the session ends.

In [ ]:
import shutil

df_sorted = df.sort_values('test_f1', ascending=False)
best_name = df_sorted.index[0]

name_to_file = {
    'Custom CNN':     'custom_cnn_final.keras',
    'ResNet50':       'resnet50_final.keras',
    'MobileNetV2':    'mobilenetv2_final.keras',
    'EfficientNetB0': 'efficientnetb0_final.keras',
}

print("Final Rankings (by F1):")
print("="*55)
for i, (name, row) in enumerate(df_sorted.iterrows(), 1):
    tag = "  ← BEST (deploying this)" if i == 1 else ""
    print(f"  {i}. {name:<18}  Acc={row['test_accuracy']:.3f}  F1={row['test_f1']:.3f}{tag}")
print("="*55)

best_info = {
    'best_model':    best_name,
    'model_file':    name_to_file[best_name],
    'test_accuracy': float(df_sorted.iloc[0]['test_accuracy']),
    'test_f1':       float(df_sorted.iloc[0]['test_f1'])
}
with open(f'{OUTPUT_DIR}/best_model.json', 'w') as f:
    json.dump(best_info, f, indent=2)

# Save best model to Drive
DRIVE_MODELS = os.path.join(BASE_DRIVE, 'saved_models')
os.makedirs(DRIVE_MODELS, exist_ok=True)
best_src = os.path.join(MODEL_DIR, name_to_file[best_name])
if os.path.exists(best_src):
    shutil.copy(best_src, os.path.join(DRIVE_MODELS, 'best_model.keras'))
    print(f"\nBest model copied to Drive → saved_models/best_model.keras")

# Copy all plots to Drive
DRIVE_OUTPUT = os.path.join(BASE_DRIVE, 'outputs')
if os.path.exists(DRIVE_OUTPUT):
    shutil.rmtree(DRIVE_OUTPUT)
shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT)
print(f"All plots copied to Drive → outputs/")

---
## Section 6 — YOLOv8 Object Detection

Classification answers *what* is in the image. Detection answers *what* and *where*. YOLOv8 draws a bounding box around each detected object and attaches a confidence score.

This section is optional for the Streamlit app to work — the classifier alone is enough. But detection adds something the classifier can't: it shows the user exactly which region of the image the model is responding to, which is both more informative and more trustworthy.

### Detection dataset structure (confirmed from your Drive)

```
object_detection_Dataset/
  train/
    images/    ← 2662 images
    labels/    ← one .txt per image
  valid/
    images/    ← 442 images
    labels/
  test/
    images/    ← 215 images
    labels/
```

Each `.txt` annotation file has one line per object:
```
class_id  cx  cy  width  height
```
All values after `class_id` are normalised to [0, 1] relative to image dimensions. `class_id=0` is bird, `class_id=1` is drone.

### YOLOv8n
We use YOLOv8 nano — smallest and fastest. If you have GPU quota to spare, change `yolov8n.pt` to `yolov8s.pt` for a noticeable accuracy bump.

### 6.1 Create data.yaml

YOLOv8 reads its dataset configuration from a YAML file. It needs the absolute path to the dataset root and relative paths to each split's `images/` folder. It finds the `labels/` folder automatically by replacing `images` with `labels` in the path — which is why the folder structure has to follow that pattern exactly.

In [ ]:
import yaml

data_yaml = {
    'path':  DETECT_DIR,
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc':    2,
    'names': ['bird', 'drone']
}

yaml_path = os.path.join(DETECT_DIR, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"data.yaml written to: {yaml_path}")
print()
print(yaml.dump(data_yaml))

### 6.2 Annotation Preview

Before training the detector, we check that the bounding box annotations line up with actual objects. We read the `.txt` files, convert the normalised coordinates back to pixel coordinates, and draw boxes on the images.

Green boxes = bird. Red boxes = drone. If boxes are wildly off, the label format or folder pairing is wrong.

In [ ]:
from pathlib import Path

def draw_yolo_boxes(img_path, label_path, class_names, ax):
    img = Image.open(img_path).convert('RGB')
    w, h = img.size
    ax.imshow(img); ax.axis('off')
    ax.set_title(Path(img_path).name, fontsize=7)
    c_map = {'bird': '#4CAF50', 'drone': '#F44336'}
    if not os.path.exists(label_path):
        return
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls_id = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:5])
            name  = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)
            color = c_map.get(name, 'yellow')
            x1 = (cx - bw/2) * w; y1 = (cy - bh/2) * h
            rect = patches.Rectangle((x1, y1), bw*w, bh*h,
                                      linewidth=2, edgecolor=color, facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1-3, name, color=color, fontsize=8, fontweight='bold')

train_img_dir = os.path.join(DETECT_DIR, 'train', 'images')
train_lbl_dir = os.path.join(DETECT_DIR, 'train', 'labels')

if os.path.exists(train_img_dir):
    img_files = [f for f in os.listdir(train_img_dir)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    sample = random.sample(img_files, min(8, len(img_files)))
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    for ax, fname in zip(axes.flatten(), sample):
        draw_yolo_boxes(
            os.path.join(train_img_dir, fname),
            os.path.join(train_lbl_dir, Path(fname).stem + '.txt'),
            CLASS_NAMES, ax
        )
    plt.suptitle('Ground-Truth Annotations — 8 Random Train Images (Green=Bird, Red=Drone)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/yolo_annotation_preview.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved.")
else:
    print(f"Not found: {train_img_dir}")

### 6.3 Train YOLOv8

The `ultralytics` library handles the full training loop — we just configure it. A few settings worth explaining:

- `imgsz=640` — YOLOv8's native resolution, higher than the classifier's 224×224 because detection needs spatial precision to draw tight boxes
- `patience=10` — stops early if mAP doesn't improve for 10 epochs
- `batch=16` — smaller than the classifier because 640×640 images are much larger per batch
- `device=0` — GPU; change to `'cpu'` if no GPU but expect very slow training

In [ ]:
from ultralytics import YOLO

yolo = YOLO('yolov8n.pt')  # nano — swap to yolov8s.pt for better accuracy

print("Training YOLOv8n...")
results = yolo.train(
    data     = yaml_path,
    epochs   = 50,
    imgsz    = 640,
    batch    = 16,
    patience = 10,
    device   = 0,
    project  = MODEL_DIR,
    name     = 'yolov8_bird_drone',
    exist_ok = True
)

YOLO_WEIGHTS = f'{MODEL_DIR}/yolov8_bird_drone/weights/best.pt'
print(f"\nDone. Best weights: {YOLO_WEIGHTS}")

### 6.4 Validate

Detection metrics are different from classification:

- **mAP@0.5** — a predicted box counts as correct if it overlaps the ground truth by at least 50% (IoU ≥ 0.5). This is the standard single-threshold metric for object detection.
- **mAP@0.5:0.95** — averaged over IoU thresholds from 0.5 to 0.95 in steps of 0.05. Much stricter — rewards models that draw tight boxes, not just roughly-right ones.

A good mAP@0.5 on a 2-class aerial dataset is roughly 0.80+. mAP@0.5:0.95 around 0.55-0.65 is reasonable for a nano model.

In [ ]:
yolo_eval = YOLO(YOLO_WEIGHTS)
val_res   = yolo_eval.val(data=yaml_path, imgsz=640)

print("Validation Results:")
print(f"  mAP@0.50      : {val_res.box.map50:.4f}")
print(f"  mAP@0.50:0.95 : {val_res.box.map:.4f}")
print(f"  Precision     : {val_res.box.mp:.4f}")
print(f"  Recall        : {val_res.box.mr:.4f}")

yolo_metrics = {
    'model':     'YOLOv8n',
    'mAP50':     float(val_res.box.map50),
    'mAP50_95':  float(val_res.box.map),
    'precision': float(val_res.box.mp),
    'recall':    float(val_res.box.mr)
}
with open(f'{OUTPUT_DIR}/yolo_metrics.json', 'w') as f:
    json.dump(yolo_metrics, f, indent=2)
print("Metrics saved.")

### 6.5 Inference on Test Images

Final check — run the trained detector on 6 random test images and draw the predicted boxes. Confidence score shown on each box is how certain the model is about that detection. We threshold at 0.4 — predictions below 40% confidence are dropped.

In [ ]:
test_img_dir = os.path.join(DETECT_DIR, 'test', 'images')
test_imgs    = [os.path.join(test_img_dir, f)
                for f in os.listdir(test_img_dir)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
sample_test  = random.sample(test_imgs, min(6, len(test_imgs)))
inf_results  = yolo_eval.predict(source=sample_test, imgsz=640, conf=0.4, save=False)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
c_map = {0: '#4CAF50', 1: '#F44336'}

for ax, res in zip(axes.flatten(), inf_results):
    img = Image.open(res.path).convert('RGB')
    ax.imshow(img); ax.axis('off')
    ax.set_title(Path(res.path).name, fontsize=7)
    if res.boxes is not None:
        for box in res.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cls_id = int(box.cls[0]); conf = float(box.conf[0])
            name   = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else str(cls_id)
            color  = c_map.get(cls_id, 'yellow')
            ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                                           linewidth=2.5, edgecolor=color, facecolor='none'))
            ax.text(x1, y1-4, f'{name} {conf:.2f}', color='white', fontsize=8,
                    fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.8))

plt.suptitle('YOLOv8n Inference — Test Images (Green=Bird, Red=Drone)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/yolo_inference.png', dpi=150, bbox_inches='tight')
plt.show()

# Copy YOLO weights to Drive
DRIVE_MODELS = os.path.join(BASE_DRIVE, 'saved_models')
os.makedirs(DRIVE_MODELS, exist_ok=True)
if os.path.exists(YOLO_WEIGHTS):
    shutil.copy(YOLO_WEIGHTS, os.path.join(DRIVE_MODELS, 'yolov8_best.pt'))
    print("YOLOv8 weights copied to Drive → saved_models/yolov8_best.pt")

# Update outputs on Drive
DRIVE_OUTPUT = os.path.join(BASE_DRIVE, 'outputs')
if os.path.exists(DRIVE_OUTPUT):
    shutil.rmtree(DRIVE_OUTPUT)
shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT)
print("Final outputs copied to Drive.")

---
## Done

Everything that was trained and generated is now in your Google Drive:

```
aerial_project/
  saved_models/
    best_model.keras       ← best classifier → goes in app/models/
    yolov8_best.pt         ← detector weights → goes in app/models/
  outputs/
    class_distribution.png
    sample_images.png
    augmentation_preview.png
    pixel_distributions.png
    image_dimensions.png
    cnn_training_curves.png
    resnet50_p1/p2_curves.png
    mobilenetv2_p1/p2_curves.png
    efficientnetb0_p1/p2_curves.png
    *_cm.png  (confusion matrix per model)
    model_comparison_bar.png
    model_radar.png
    yolo_annotation_preview.png
    yolo_inference.png
    best_model.json
```

### Running the Streamlit app locally

```bash
# Download best_model.keras and yolov8_best.pt from Drive
# Place both inside a models/ folder next to app.py

pip install -r requirements.txt
streamlit run app.py
```